In [1]:
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import gc

@dataclass
class SequenceStore:
    # Per-row arrays
    species_ids: np.ndarray
    gene_ids: np.ndarray
    sequences: list[np.ndarray]
    # Lookup tables
    species_to_gene_to_rows: dict[int, dict[int, list[int]]]
    valid_species: np.ndarray
    # Name lookups
    species_names: list[str]
    gene_names: list[str]
    # Token map used for pre-encoding
    token_to_id: dict[str, int]

    @classmethod
    def from_parquet(cls, parquet_path: str | Path, species_col: str = "species", gene_col: str = "gene", 
                     sequence_col: str = "sequence", keep_n_species: int | None = None) -> "SequenceStore":
        print("[SequenceStore] - Loading Parquet file")
        df = pd.read_parquet(parquet_path, columns=[species_col, gene_col, sequence_col])

        # Basic checks
        print("[SequenceStore] - Checking for missing values")
        if df[species_col].isna().any():
            raise ValueError(f"Column '{species_col}' contains missing values.")
        if df[gene_col].isna().any():
            raise ValueError(f"Column '{gene_col}' contains missing values.")
        if df[sequence_col].isna().any():
            raise ValueError(f"Column '{sequence_col}' contains missing values.")
        
        # Ensure categorical dtype, and remove unused categories to ensure continuous numbering
        print("[SequenceStore] - Ensuring continuous categorical species and gene columns")
        species_cat = df[species_col].astype('category').cat.remove_unused_categories()
        gene_cat = df[gene_col].astype('category').cat.remove_unused_categories()

        if keep_n_species is not None and keep_n_species > 0:
            keep_species_ids = species_cat.cat.categories[:keep_n_species]
            species_cat = species_cat[species_cat.isin(keep_species_ids)]

        # Extract category integer IDs
        print("[SequenceStore] - Extracting category integer IDs")
        species_ids = species_cat.cat.codes.to_numpy(copy=True)
        gene_ids = gene_cat.cat.codes.to_numpy(copy=True)

        # Extract category names
        print("[SequenceStore] - Extracting category names")
        species_names = species_cat.cat.categories.to_list()
        gene_names = gene_cat.cat.categories.to_list()

        #Pre-encode sequences, using tiny vocabulary for DNA / ambiguous bases
        print("[SequenceStore] - Pre-encoding sequences, using tiny vocabulary for DNA / ambiguous bases ")
        token_to_id = {
            'PAD': 0,
            'A': 1,
            'C': 2,
            'G': 3,
            'T': 4,
            'N': 5,
        }

        encode_array = np.full(256, token_to_id['N'], dtype=np.uint8)
        for token, idx in token_to_id.items():
            if token != 'PAD':
                encode_array[ord(token)] = idx

        def encode_dna(seq):
        # Convert string to ASCII bytes, then use the encode_array as a lookup table
            return encode_array[np.frombuffer(seq.encode('ascii'), dtype=np.uint8)]
        
        sequences = [encode_dna(seq) for seq in df[sequence_col].array]
        
        # Build nested row-index mapping
        print("[SequenceStore] - Building nested row-index mapping")
        species_to_gene_to_rows = defaultdict(lambda: defaultdict(list))
        for row_idx, (sid, gid) in enumerate(zip(species_ids, gene_ids)):
            species_to_gene_to_rows[int(sid)][int(gid)].append(row_idx)

        species_to_gene_to_rows = {
            sid: dict(gene_map)
            for sid, gene_map in species_to_gene_to_rows.items()
        }

        valid_species = np.array(sorted(species_to_gene_to_rows.keys()))

        print("[SequenceStore] - Cleaning up temporary memory")
        del df, species_cat, gene_cat
        gc.collect()

        return cls(
            species_ids=species_ids,
            gene_ids=gene_ids,
            sequences=sequences,
            species_to_gene_to_rows=species_to_gene_to_rows,
            valid_species=valid_species,
            species_names=species_names,
            gene_names=gene_names,
            token_to_id=token_to_id
        )
    
    @property
    def num_rows(self) -> int:
        return len(self.sequences)
    
    @property
    def num_species(self) -> int:
        return len(self.species_names)
    
    @property
    def num_valid_species(self) -> int:
        return len(self.valid_species)
    
    @property
    def num_genes(self) -> int:
        return len(self.gene_names)
    
    @property
    def vocab_size(self) -> int:
        return len(self.token_to_id)
    
    def summary(self) -> str:
        return (
            f"SequenceStore(num_rows={self.num_rows}, "
            f"num_species={self.num_species}, "
            f"num_genes={self.num_genes}, "
            f"valid_species={self.num_valid_species})"
        )

In [2]:
path_processed_dataset = Path("output") / "processed_dataset.parquet"
store = SequenceStore.from_parquet(path_processed_dataset)
store.summary()

[SequenceStore] - Loading Parquet file
[SequenceStore] - Checking for missing values
[SequenceStore] - Ensuring continuous categorical species and gene columns
[SequenceStore] - Extracting category integer IDs
[SequenceStore] - Extracting category names
[SequenceStore] - Pre-encoding sequences, using tiny vocabulary for DNA / ambiguous bases 
[SequenceStore] - Building nested row-index mapping
[SequenceStore] - Cleaning up temporary memory


'SequenceStore(num_rows=472855, num_species=11457, num_genes=15, valid_species=11457)'

In [3]:
import torch


@dataclass
class BatchBuilder:
    store: SequenceStore
    species_per_batch: int = 64
    crop_length: int = 512
    rng_seed: int | None = None
    pin_memory: bool = False

    def __post_init__(self) -> None:
        self.rng = np.random.default_rng(self.rng_seed)
        self.pad_id = self.store.token_to_id["PAD"]

        if self.species_per_batch < 1:
            raise ValueError("species_per_batch must be at least 1.")
        if self.crop_length < 1:
            raise ValueError("crop_length must be at least 1.")
        if self.species_per_batch > len(self.store.valid_species):
            raise ValueError(
                "species_per_batch cannot exceed the number of valid species."
            )

    @property
    def batch_size(self) -> int:
        return 2 * self.species_per_batch

    def sample_species(self) -> np.ndarray:
        return self.rng.choice(
            self.store.valid_species,
            size=self.species_per_batch,
            replace=False,
        )

    def sample_two_rows_for_species(self, species_id: int) -> tuple[tuple[int, int], tuple[int, int]]:
        gene_to_rows = self.store.species_to_gene_to_rows[species_id]
        gene_ids = list(gene_to_rows.keys())

        # Case A: species has >= 2 genes
        if len(gene_ids) >= 2:
            gene1, gene2 = self.rng.choice(gene_ids, size=2, replace=False)
            row1 = int(self.rng.choice(gene_to_rows[gene1]))
            row2 = int(self.rng.choice(gene_to_rows[gene2]))
            return (row1, gene1), (row2, gene2)

        # Only one gene available
        gene = gene_ids[0]
        rows = gene_to_rows[gene]

        # Case B: one gene, >= 2 rows
        if len(rows) >= 2:
            row1, row2 = self.rng.choice(rows, size=2, replace=False)
            return (int(row1), gene), (int(row2), gene)

        # Case C: one gene, one row
        row = rows[0]
        return (row, gene), (row, gene)

    def write_crop_into_cpu_tensors(self, seq: np.ndarray, input_ids: torch.Tensor, 
                                    attention_mask: torch.Tensor, batch_idx: int) -> None:
        L = self.crop_length
        seq_len = len(seq)

        if seq_len >= L:
            if seq_len == L:
                crop = seq
            else:
                start = self.rng.integers(0, seq_len - L + 1)
                crop = seq[start:start + L]

            # copy crop into CPU tensor row
            input_ids[batch_idx].copy_(torch.from_numpy(crop.astype(np.int64, copy=False)))
            attention_mask[batch_idx].fill_(True)
            return

        # Shorter than crop length: pad
        input_ids[batch_idx].fill_(self.pad_id)
        attention_mask[batch_idx].fill_(False)

        input_ids[batch_idx, :seq_len].copy_(
            torch.from_numpy(seq.astype(np.int64, copy=False))
        )
        attention_mask[batch_idx, :seq_len] = True

    def sample_batch_cpu(self) -> dict[str, torch.Tensor]:
        sampled_species = self.sample_species()
        B = self.batch_size
        L = self.crop_length

        input_ids = torch.full((B, L), fill_value=self.pad_id, dtype=torch.long, device="cpu")
        attention_mask = torch.zeros((B, L), dtype=torch.bool, device="cpu")
        species_ids = torch.empty(B, dtype=torch.long, device="cpu")
        gene_ids = torch.empty(B, dtype=torch.long, device="cpu")

        batch_idx = 0
        for species_id in sampled_species:
            species_id = int(species_id)
            view1, view2 = self.sample_two_rows_for_species(species_id)

            for row_idx, gene_id in (view1, view2):
                seq = self.store.sequences[row_idx]
                self.write_crop_into_cpu_tensors(
                    seq=seq,
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    batch_idx=batch_idx,
                )
                species_ids[batch_idx] = species_id
                gene_ids[batch_idx] = int(gene_id)
                batch_idx += 1

        if self.pin_memory and torch.cuda.is_available():
            input_ids = input_ids.pin_memory()
            attention_mask = attention_mask.pin_memory()
            species_ids = species_ids.pin_memory()
            gene_ids = gene_ids.pin_memory()

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "species_ids": species_ids,
            "gene_ids": gene_ids,
        }

    def move_batch_to_device(
        self,
        batch: dict[str, torch.Tensor],
        device: str | torch.device,
        non_blocking: bool = True,
    ) -> dict[str, torch.Tensor]:
        """
        Move a CPU batch to the target device.
        """
        device = torch.device(device)
        return {
            k: v.to(device, non_blocking=non_blocking)
            for k, v in batch.items()
        }

In [4]:
builder = BatchBuilder(
    store=store,
    species_per_batch=64,
    crop_length=512,
    rng_seed=None,
    pin_memory=True
)

batch = builder.sample_batch_cpu()

for i in range(0, len(batch["species_ids"]), 2):
    sid1 = int(batch["species_ids"][i].item())
    sid2 = int(batch["species_ids"][i + 1].item())
    gid1 = int(batch["gene_ids"][i].item())
    gid2 = int(batch["gene_ids"][i + 1].item())

    print(
        f"pair {i//2}: species={store.species_names[sid1]}, "
        f"genes=({store.gene_names[gid1]}, {store.gene_names[gid2]})"
    )

pair 0: species=Acanthemblemaria chaplini, genes=(ATPase6, COXI)
pair 1: species=Parabotia kiangsiensis, genes=(ATPase8, ND3)
pair 2: species=Chaeturichthys stigmatias, genes=(ATPase6, ND2)
pair 3: species=Rhadinocentrus ornatus, genes=(COXI, 12S_rRNA)
pair 4: species=Etheostoma blennioides, genes=(COXIII, ATPase6)
pair 5: species=Nangra assamensis, genes=(COXI, COXI)
pair 6: species=Hemiramphus archipelagicus, genes=(16S_rRNA, Cytb)
pair 7: species=Alburnus arborella, genes=(COXI, 12S_rRNA)
pair 8: species=Parascombrops spinosus, genes=(ND5, COXIII)
pair 9: species=Hemiancistrus maracaiboensis, genes=(Cytb, Cytb)
pair 10: species=Moenkhausia xinguensis, genes=(16S_rRNA, Cytb)
pair 11: species=Saurida gracilis, genes=(ND3, ND2)
pair 12: species=Odontanthias elizabethae, genes=(COXI, COXI)
pair 13: species=Pseudostegophilus nemurus, genes=(ND4, COXI)
pair 14: species=Poeciliopsis hnilickai, genes=(ND1, ND2)
pair 15: species=Trichopodus pectoralis, genes=(ATPase6, 16S_rRNA)
pair 16: spec

In [ ]:
import torch.nn as nn
import torch.nn.functional as F


class SequenceEncoder(nn.Module):
    """
    Transformer-based encoder for DNA sequences with optional gene embeddings.

    Input:
        input_ids:      [B, L] torch.long
        attention_mask: [B, L] torch.bool
            True  = real token
            False = padding
        gene_ids:       [B] torch.long
            gene identity for each sequence

    Output:
        z_seq: [B, embed_dim]
            L2-normalized sequence embeddings
    """

    def __init__(self, vocab_size: int, max_length: int, num_genes: int, d_model: int = 256, nhead: int = 8,
                 num_layers: int = 4, dim_feedforward: int = 1024, dropout: float = 0.1, embed_dim: int = 256,
                 pad_token_id: int = 0, use_gene_embedding: bool = True) -> None:
        super().__init__()

        if d_model % nhead != 0:
            raise ValueError("d_model must be divisible by nhead.")

        self.vocab_size = vocab_size
        self.max_length = max_length
        self.num_genes = num_genes
        self.d_model = d_model
        self.embed_dim = embed_dim
        self.pad_token_id = pad_token_id
        self.use_gene_embedding = use_gene_embedding

        # Token embedding
        self.token_embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=d_model,
            padding_idx=pad_token_id,
        )

        # Positional embedding
        self.position_embedding = nn.Embedding(
            num_embeddings=max_length,
            embedding_dim=d_model,
        )

        # Optional gene embedding
        if self.use_gene_embedding:
            self.gene_embedding = nn.Embedding(
                num_embeddings=num_genes,
                embedding_dim=d_model,
            )

        # Input dropout
        self.input_dropout = nn.Dropout(dropout)

        # Transformer encoder stack
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer=encoder_layer,
            num_layers=num_layers,
        )

        # Projection head
        self.projection = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, embed_dim),
        )

    def masked_mean_pool(self, x: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        """
        x: [B, L, d_model]
        attention_mask: [B, L] bool, True=real token, False=padding
        """
        mask = attention_mask.unsqueeze(-1).to(dtype=x.dtype)  # [B, L, 1]
        x_masked = x * mask
        summed = x_masked.sum(dim=1)  # [B, d_model]
        counts = mask.sum(dim=1).clamp(min=1.0)  # [B, 1]
        return summed / counts

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor,
                gene_ids: torch.Tensor | None = None) -> torch.Tensor:
        """
        input_ids:      [B, L] torch.long
        attention_mask: [B, L] torch.bool
        gene_ids:       [B] torch.long or None

        returns:
            z_seq: [B, embed_dim], L2-normalized
        """
        if input_ids.ndim != 2:
            raise ValueError(f"input_ids must have shape [B, L], got {input_ids.shape}")
        if attention_mask.ndim != 2:
            raise ValueError(
                f"attention_mask must have shape [B, L], got {attention_mask.shape}"
            )
        if input_ids.shape != attention_mask.shape:
            raise ValueError(
                f"input_ids and attention_mask must have same shape, got "
                f"{input_ids.shape} and {attention_mask.shape}"
            )

        batch_size, seq_len = input_ids.shape

        if seq_len > self.max_length:
            raise ValueError(
                f"Sequence length {seq_len} exceeds max_length={self.max_length}"
            )

        if self.use_gene_embedding:
            if gene_ids is None:
                raise ValueError("gene_ids must be provided when use_gene_embedding=True")
            if gene_ids.ndim != 1 or gene_ids.shape[0] != batch_size:
                raise ValueError(
                    f"gene_ids must have shape [B], got {gene_ids.shape}"
                )

        # Position indices: [B, L]
        positions = torch.arange(seq_len, device=input_ids.device).unsqueeze(0).expand(
            batch_size, seq_len
        )

        # Token + position embeddings
        x = self.token_embedding(input_ids) + self.position_embedding(positions)

        # Add gene embedding if enabled
        if self.use_gene_embedding:
            gene_vec = self.gene_embedding(gene_ids)          # [B, d_model]
            gene_vec = gene_vec.unsqueeze(1)                  # [B, 1, d_model]
            x = x + gene_vec                                  # broadcast over L

        # Input dropout
        x = self.input_dropout(x)

        # PyTorch transformer expects True for positions to ignore
        src_key_padding_mask = ~attention_mask  # [B, L]

        x = self.transformer(x, src_key_padding_mask=src_key_padding_mask)  # [B, L, d_model]

        pooled = self.masked_mean_pool(x, attention_mask)    # [B, d_model]
        z_seq = self.projection(pooled)                      # [B, embed_dim]
        z_seq = F.normalize(z_seq, dim=-1)

        return z_seq

In [6]:
class SpeciesContrastiveModel(nn.Module):
    """
    Sequence encoder + global species prototype table.

    Training objective:
        each sequence embedding should be close to the correct species
        embedding and far from all other species embeddings.
    """
    def __init__(self, encoder: SequenceEncoder, num_species: int, embed_dim: int, temperature: float = 0.07,
                 lambda_proto: float = 1.0, lambda_seq: float = 1.0) -> None:
        super().__init__()

        if temperature <= 0:
            raise ValueError("temperature must be > 0")

        self.encoder = encoder
        self.temperature = temperature
        self.lambda_proto = lambda_proto
        self.lambda_seq = lambda_seq

        self.species_embedding = nn.Embedding(num_species, embed_dim)
    
        
    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor, species_ids: torch.Tensor,
                gene_ids: torch.Tensor) -> tuple[torch.Tensor, dict[str, torch.Tensor]]:
        """
        Loss = batch-restricted prototype loss + sequence-sequence auxiliary loss

        Assumes the batch is organized in paired views:
            0,1   = same species
            2,3   = same species
            4,5   = same species
            ...

        Parameters
        ----------
        input_ids : [B, L]
        attention_mask : [B, L]
        species_ids : [B]

        Returns
        -------
        total_loss : scalar tensor
        outputs : dict of tensors for logging/debugging
        """
        # 1) Encode sequences
        z_seq = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            gene_ids=gene_ids
        )  # [B, D], already normalized by encoder

        B = z_seq.shape[0]
        device = z_seq.device

        # 2) Batch-restricted prototype loss
        # Get unique species in this batch, and map each example to its local index
        batch_species_ids, local_targets = torch.unique(
            species_ids,
            sorted=True,
            return_inverse=True,
        )  # batch_species_ids: [S], local_targets: [B]

        z_species_batch = self.species_embedding(batch_species_ids)  # [S, D]
        z_species_batch = F.normalize(z_species_batch, dim=-1)

        logits_proto = (z_seq @ z_species_batch.T) / self.temperature  # [B, S]
        loss_proto = F.cross_entropy(logits_proto, local_targets)

        # 3) Sequence-sequence auxiliary contrastive loss
        logits_seq = (z_seq @ z_seq.T) / self.temperature  # [B, B]

        # mask self-similarity so each sequence cannot choose itself
        eye_mask = torch.eye(B, dtype=torch.bool, device=device)
        logits_seq = logits_seq.masked_fill(eye_mask, -1e9)

        # paired-view targets:
        # 0 <-> 1, 2 <-> 3, 4 <-> 5, ...
        # x ^ 1 flips the last bit:
        # 0->1, 1->0, 2->3, 3->2, ...
        targets_seq = torch.arange(B, device=device) ^ 1

        loss_seq = F.cross_entropy(logits_seq, targets_seq)

        # 4) Combine losses
        total_loss = self.lambda_proto * loss_proto + self.lambda_seq * loss_seq

        outputs = {
            "z_seq": z_seq,
            "batch_species_ids": batch_species_ids,   # [S]
            "z_species_batch": z_species_batch,       # [S, D]
            "logits_proto": logits_proto,             # [B, S]
            "logits_seq": logits_seq,                 # [B, B]
            "loss_proto": loss_proto.detach(),
            "loss_seq": loss_seq.detach(),
            "local_targets": local_targets,           # [B]
            "targets_seq": targets_seq,               # [B]
        }

        return total_loss, outputs

In [7]:
import time

def topk_accuracy(logits: torch.Tensor, targets: torch.Tensor, k: int = 1) -> torch.Tensor:
    """
    Compute top-k accuracy.

    logits:  [B, C]
    targets: [B]
    """
    k = min(k, logits.shape[1])
    topk = logits.topk(k, dim=1).indices
    correct = (topk == targets.unsqueeze(1)).any(dim=1)
    return correct.float().mean()


def save_checkpoint(save_path: str | Path, model: SpeciesContrastiveModel, optimizer: torch.optim.Optimizer,
    step: int, loss: float, loss_proto: float, loss_seq: float, acc_proto_top1: float, acc_proto_top5: float,
    acc_seq: float, species_names: list[str]) -> None:
    """
    Save a training checkpoint including model weights and the current
    species embedding matrix for downstream clustering analysis.
    """
    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)

    checkpoint = {
        "step": step,
        "loss": loss,
        "loss_proto": loss_proto,
        "loss_seq": loss_seq,
        "acc_proto_top1": acc_proto_top1,
        "acc_proto_top5": acc_proto_top5,
        "acc_seq": acc_seq,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "species_embeddings": model.species_embedding.weight.detach().cpu(),
        "species_names": species_names,
    }

    torch.save(checkpoint, save_path)


def train_species_contrastive(model: SpeciesContrastiveModel, builder: BatchBuilder, optimizer: torch.optim.Optimizer,
    device: str | torch.device, num_steps: int, checkpoint_dir: str | Path, checkpoint_every: int = 500,
    log_every: int = 50, grad_clip_norm: float | None = 1.0) -> list[dict]:
    """
    Train the species-contrastive model.

    Periodically saves checkpoints containing:
    - encoder + species embedding weights
    - optimizer state
    - current species embedding matrix
    - metadata (step, losses, accuracies)

    Returns
    -------
    history : list of dict
        Per-log-step training metrics.
    """
    device = torch.device(device)
    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    model.train()
    history = []

    t0 = time.time()

    for step in range(1, num_steps + 1):
        # Build batch on CPU, then move once to GPU
        batch_cpu = builder.sample_batch_cpu()
        batch = builder.move_batch_to_device(batch_cpu, device=device, non_blocking=True)

        # Optional debugging assertion: paired views should have same species
        assert torch.all(
            batch["species_ids"][0::2] == batch["species_ids"][1::2]
        ), "Batch pairs are not aligned by species."

        loss, outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            species_ids=batch["species_ids"],
            gene_ids=batch["gene_ids"]
        )

        logits_proto = outputs["logits_proto"]
        logits_seq = outputs["logits_seq"]
        local_targets = outputs["local_targets"]
        targets_seq = outputs["targets_seq"]

        acc_proto_top1 = topk_accuracy(logits_proto, local_targets, k=1)
        acc_proto_top5 = topk_accuracy(logits_proto, local_targets, k=5)
        acc_seq = topk_accuracy(logits_seq, targets_seq, k=1)

        loss_proto = outputs["loss_proto"]
        loss_seq = outputs["loss_seq"]

        optimizer.zero_grad(set_to_none=True)
        loss.backward()

        if grad_clip_norm is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)

        optimizer.step()

        # Logging
        if step % log_every == 0 or step == 1:
            elapsed = time.time() - t0
            metrics = {
                "step": step,
                "loss": float(loss.item()),
                "loss_proto": float(loss_proto.item()),
                "loss_seq": float(loss_seq.item()),
                "acc_proto_top1": float(acc_proto_top1.item()),
                "acc_proto_top5": float(acc_proto_top5.item()),
                "acc_seq": float(acc_seq.item()),
                "elapsed_sec": elapsed,
            }
            history.append(metrics)

            print(
                f"[step {step:6d}] "
                f"loss={metrics['loss']:.4f} "
                f"proto={metrics['loss_proto']:.4f} "
                f"seq={metrics['loss_seq']:.4f} "
                f"acc1={metrics['acc_proto_top1']:.4f} "
                f"acc5={metrics['acc_proto_top5']:.4f} "
                f"acc_seq={metrics['acc_seq']:.4f} "
                f"time={elapsed:.1f}s"
            )

        # Periodic checkpointing
        if step % checkpoint_every == 0 or step == num_steps:
            save_path = checkpoint_dir / f"step_{step:07d}.pt"
            save_checkpoint(
                save_path=save_path,
                model=model,
                optimizer=optimizer,
                step=step,
                loss=float(loss.item()),
                loss_proto=float(loss_proto.item()),
                loss_seq=float(loss_seq.item()),
                acc_proto_top1=float(acc_proto_top1.item()),
                acc_proto_top5=float(acc_proto_top5.item()),
                acc_seq=float(acc_seq.item()),
                species_names=builder.store.species_names,
            )
            print(f"Saved checkpoint to {save_path}")

    return history

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

encoder = SequenceEncoder(
    vocab_size=store.vocab_size,
    max_length=builder.crop_length,
    num_genes=store.num_genes,
    d_model=256,
    nhead=8,
    num_layers=4,
    dim_feedforward=1024,
    dropout=0.1,
    embed_dim=256,
    pad_token_id=store.token_to_id["PAD"],
    use_gene_embedding=True
).to(device)

model = SpeciesContrastiveModel(
    encoder=encoder,
    num_species=store.num_valid_species,
    embed_dim=256,
    temperature=0.07,
    lambda_proto=1.0,
    lambda_seq=1.0
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-2,
)

history = train_species_contrastive(
    model=model,
    builder=builder,
    optimizer=optimizer,
    device=device,
    num_steps=10_000,
    checkpoint_dir=Path("output") / "checkpoints_species_contrastive",
    checkpoint_every=500,
    log_every=50,
    grad_clip_norm=1.0,
)

/tmp/ipykernel_99572/2080542408.py:68: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


[step      1] loss=14.4722 proto=4.5526 seq=9.9197 acc1=0.0156 acc5=0.1172 acc_seq=0.0000 time=0.8s
[step     50] loss=9.2435 proto=4.4562 seq=4.7873 acc1=0.0156 acc5=0.0859 acc_seq=0.0078 time=28.8s
[step    100] loss=9.3968 proto=4.4499 seq=4.9468 acc1=0.0156 acc5=0.0938 acc_seq=0.0078 time=58.0s
[step    150] loss=9.1891 proto=4.5069 seq=4.6822 acc1=0.0156 acc5=0.0625 acc_seq=0.0078 time=87.2s
[step    200] loss=9.1995 proto=4.4653 seq=4.7342 acc1=0.0156 acc5=0.0781 acc_seq=0.0078 time=116.4s
[step    250] loss=9.2562 proto=4.5387 seq=4.7175 acc1=0.0234 acc5=0.0703 acc_seq=0.0078 time=145.7s
[step    300] loss=9.1967 proto=4.4458 seq=4.7509 acc1=0.0156 acc5=0.0625 acc_seq=0.0156 time=175.0s
[step    350] loss=9.2708 proto=4.4979 seq=4.7729 acc1=0.0234 acc5=0.0781 acc_seq=0.0000 time=204.2s
[step    400] loss=9.2383 proto=4.4632 seq=4.7751 acc1=0.0156 acc5=0.0938 acc_seq=0.0469 time=233.5s
[step    450] loss=9.1973 proto=4.3722 seq=4.8251 acc1=0.0078 acc5=0.0781 acc_seq=0.0078 time=2

KeyboardInterrupt: 